# Set Library Path

In [1]:
.libPaths("/home/vhaghani/anaconda3/envs/jupyter_nb_R4.4.2/lib/R/library")

# Load Libraries

In [3]:
library(enrichR)
library(dplyr)
library(openxlsx)
library(ggplot2)
library(readxl)
library(tidyr)

# Load Data

In [4]:
# List all DEG files in the subdirectories 
files <- list.files(path = "05_DEGs", pattern = "_results.csv$", full.names = TRUE, recursive = TRUE)

# Initialize an empty vector to hold files with significant DEGs
valid_files <- c()

# Loop through each file
for (file in files) {
  
  # Read the current file
  df <- read.csv(file, row.names = 1)
  
  # Extract comparison name 
  comparison <- gsub("_results.csv", "", basename(file))
  
  # Filter to only significant DEGs (adj.P.Val < 0.05)
  significant_df <- df[df$adj.P.Val < 0.05 & !is.na(df$adj.P.Val), ]
  
  # Check the number of significant DEGs
  num_significant <- nrow(significant_df)
  
  print(paste("File:", basename(file), "- Number of significant DEGs:", num_significant))
  
  # If there are significant DEGs, keep the file
  if (num_significant > 0) {
    valid_files <- c(valid_files, file)
  }
}

# Update the original files list to only include valid files
files <- valid_files

[1] "File: R270X_D18_vs_R270X_D46_results.csv - Number of significant DEGs: 9275"
[1] "File: R270X_D18_vs_R270X_D90_results.csv - Number of significant DEGs: 11725"
[1] "File: R270X_D18_vs_WT_D18_results.csv - Number of significant DEGs: 130"
[1] "File: R270X_D18_vs_WT_D46_results.csv - Number of significant DEGs: 9315"
[1] "File: R270X_D18_vs_WT_D90_results.csv - Number of significant DEGs: 11562"
[1] "File: R270X_D46_vs_R270X_D90_results.csv - Number of significant DEGs: 4900"
[1] "File: R270X_D46_vs_WT_D46_results.csv - Number of significant DEGs: 91"
[1] "File: R270X_D46_vs_WT_D90_results.csv - Number of significant DEGs: 4960"
[1] "File: R270X_D90_vs_WT_D90_results.csv - Number of significant DEGs: 66"
[1] "File: WT_D18_vs_R270X_D46_results.csv - Number of significant DEGs: 9422"
[1] "File: WT_D18_vs_R270X_D90_results.csv - Number of significant DEGs: 11704"
[1] "File: WT_D18_vs_WT_D46_results.csv - Number of significant DEGs: 9438"
[1] "File: WT_D18_vs_WT_D90_results.csv - Number

# Plot DEGs

## Gene Ontology

In [5]:
# Loop through each file
for (file in files) {
  
  # Read the current file
  df <- read.csv(file)

  # Filter to only significant DEGs
  df <- df[df$adj.P.Val < 0.05, ]

  # Sort by adjusted p-value/logFC and take the top 500
  df <- df[order(df$adj.P.Val), ]
  df <- df[order(abs(df$logFC), decreasing = TRUE), ]
  df <- head(df, 500)
    
  # Perform the enrichR analysis on the gene list
  enrichr_results <- enrichr(df$X, c("GO_Biological_Process_2025",
                                                      "GO_Cellular_Component_2025",
                                                      "GO_Molecular_Function_2025",
                                                      "KEGG_2021_Human",
                                                      "Panther_2016",
                                                      "Reactome_2022",
                                                      "RNA-Seq_Disease_Gene_and_Drug_Signatures_from_GEO"))
  
  # Create a new Excel workbook for the current file
  wb <- createWorkbook()
  
  # Loop through each table in enrichr_results and save it as a separate sheet in the Excel workbook
  for (i in seq_along(enrichr_results)) {
    # Extract the data frame from the list
    enrichr_df <- enrichr_results[[i]]
    
    # Define the original sheet name
    original_sheet_name <- names(enrichr_results)[i]
    
    # Modify the sheet name if it's specifically "RNA-Seq_Disease_Gene_and_Drug_Signatures_from_GEO"
    sheet_name <- if (original_sheet_name == "RNA-Seq_Disease_Gene_and_Drug_Signatures_from_GEO") {
      "RNAseq_DiseaseGene_DrugSigs_GEO"
    } else {
      original_sheet_name
    }
    
    # Add the data frame as a new sheet in the Excel workbook
    addWorksheet(wb, sheet_name)
    writeData(wb, sheet = sheet_name, x = enrichr_df)
  }
    
  # Define the output file name
  output_filename <- paste0("06_DEG_GO/", tools::file_path_sans_ext(basename(file)), "_enrichr_results.xlsx")
  
  # Save the Excel workbook for this file
  saveWorkbook(wb, output_filename, overwrite = TRUE)
  
  # Print message indicating successful save
  cat("Enrichr results saved for:", file, "->", output_filename, "\n")
}

Uploading data to Enrichr... Done.
  Querying GO_Biological_Process_2025... Done.
  Querying GO_Cellular_Component_2025... Done.
  Querying GO_Molecular_Function_2025... Done.
  Querying KEGG_2021_Human... Done.
  Querying Panther_2016... Done.
  Querying Reactome_2022... Done.
  Querying RNA-Seq_Disease_Gene_and_Drug_Signatures_from_GEO... Done.
Parsing results... Done.
Enrichr results saved for: 05_DEGs/R270X_D18_vs_R270X_D46_results.csv -> 06_DEG_GO/R270X_D18_vs_R270X_D46_results_enrichr_results.xlsx 
Uploading data to Enrichr... Done.
  Querying GO_Biological_Process_2025... Done.
  Querying GO_Cellular_Component_2025... Done.
  Querying GO_Molecular_Function_2025... Done.
  Querying KEGG_2021_Human... Done.
  Querying Panther_2016... Done.
  Querying Reactome_2022... Done.
  Querying RNA-Seq_Disease_Gene_and_Drug_Signatures_from_GEO... Done.
Parsing results... Done.
Enrichr results saved for: 05_DEGs/R270X_D18_vs_R270X_D90_results.csv -> 06_DEG_GO/R270X_D18_vs_R270X_D90_results_en

## Plot All Genes

In [6]:
# Define functions to calculate -log10 of P-value and filter top 10 terms
calculate_neg_log_p <- function(df) {
  df %>%
    mutate(neg_log_p = ifelse(Adjusted.P.value > 0, -log10(Adjusted.P.value), Inf))
}

filter_top_10 <- function(df) {
  return(df %>% arrange(Adjusted.P.value) %>% head(10))
}

In [7]:
# List all Enrichr Excel files
files <- list.files(path = "06_DEG_GO", pattern = "\\.xlsx$", full.names = TRUE, recursive = TRUE)

# Loop through each Excel file and make graphs
for (file in files) {
  # Extract the base filename without extension
  filename <- tools::file_path_sans_ext(basename(file))
  
  # Get the gene name from the directory path
  gene_name <- basename(dirname(file))
  
  # Extract comparison name from filename
  comparison_name <- gsub("_enrichr_results", "", filename)
  comparison_name <- gsub("_", " ", comparison_name)
  comparison_name <- gsub("results", "", comparison_name)
  
  cat("\n=== ", gene_name, " ===\n")
  cat("Comparison:", comparison_name, "\n")
  
  # Read Enrichr results from Excel sheets
  enrichr_results <- list(
    GO_Biological_Process_2025 = read_excel(file, sheet = "GO_Biological_Process_2025"),
    GO_Cellular_Component_2025 = read_excel(file, sheet = "GO_Cellular_Component_2025"),
    GO_Molecular_Function_2025 = read_excel(file, sheet = "GO_Molecular_Function_2025")
  )
    
  # Process each GO category
  GO_BP <- enrichr_results$GO_Biological_Process_2025
  GO_CC <- enrichr_results$GO_Cellular_Component_2025
  GO_MF <- enrichr_results$GO_Molecular_Function_2025
  
  # Filter top 10 terms by p-value
  GO_BP <- GO_BP %>% arrange(Adjusted.P.value) %>% head(10)
  GO_CC <- GO_CC %>% arrange(Adjusted.P.value) %>% head(10)
  GO_MF <- GO_MF %>% arrange(Adjusted.P.value) %>% head(10)
  
  # Calculate -log10 p-values
  GO_BP$neg_log_p <- -log10(GO_BP$Adjusted.P.value)
  GO_CC$neg_log_p <- -log10(GO_CC$Adjusted.P.value)
  GO_MF$neg_log_p <- -log10(GO_MF$Adjusted.P.value)

  # Combine data frames for plotting
  combined_df <- rbind(
    mutate(GO_BP, Category = "Biological Process"),
    mutate(GO_CC, Category = "Cellular Component"),
    mutate(GO_MF, Category = "Molecular Function")
  )
    
  # Skip if no data
  if (nrow(combined_df) == 0) {
    cat("No Enrichr results for this comparison\n")
    next
  }
       
  # Plot
  pdf(paste0("00_FIGURES/EnrichR_Barplots/", comparison_name, "_enrichr_barplot.pdf"), height = 5, width = 12)

  p <- ggplot(combined_df, aes(x = neg_log_p, y = reorder(Term, neg_log_p), fill = Category)) +
    geom_col(position = position_dodge(width = 0.9)) +
    # Reference line for adjusted p-value = 0.05
    geom_vline(xintercept = -log10(0.05), linetype = "dashed", color = "lightgrey", linewidth = 0.6) +
    scale_fill_manual(values = c("Biological Process" = "#2B79B2", 
                                  "Cellular Component" = "#FA7C1A", 
                                  "Molecular Function" = "#3CA036")) +
    labs(x = "-log10(Adjusted.P.value)", y = NULL, 
         title = paste("Enrichr Results:", comparison_name)) +
    facet_grid(Category ~ ., scales = "free_y", space = "free_y") +
    theme_minimal() +
    theme(legend.position = "right",
          strip.text.y = element_blank(), 
          panel.grid.major = element_blank(),  
          panel.grid.minor = element_blank(),
          axis.line.x = element_line(color = "black", linewidth = 0.5),
          axis.line.y = element_line(color = "black", linewidth = 0.5))

  print(p)
    
  dev.off()

  # Print results
  cat("Biological Process terms:", nrow(GO_BP), "\n")
  cat("Cellular Component terms:", nrow(GO_CC), "\n")
  cat("Molecular Function terms:", nrow(GO_MF), "\n")
  cat("Plot generated for:", comparison_name, "\n")
}


===  06_DEG_GO  ===
Comparison: R270X D18 vs R270X D46  
Biological Process terms: 10 
Cellular Component terms: 10 
Molecular Function terms: 10 
Plot generated for: R270X D18 vs R270X D46  

===  06_DEG_GO  ===
Comparison: R270X D18 vs R270X D90  
Biological Process terms: 10 
Cellular Component terms: 10 
Molecular Function terms: 10 
Plot generated for: R270X D18 vs R270X D90  

===  06_DEG_GO  ===
Comparison: R270X D18 vs WT D18  
Biological Process terms: 10 
Cellular Component terms: 9 
Molecular Function terms: 10 
Plot generated for: R270X D18 vs WT D18  

===  06_DEG_GO  ===
Comparison: R270X D18 vs WT D46  
Biological Process terms: 10 
Cellular Component terms: 10 
Molecular Function terms: 10 
Plot generated for: R270X D18 vs WT D46  

===  06_DEG_GO  ===
Comparison: R270X D18 vs WT D90  
Biological Process terms: 10 
Cellular Component terms: 10 
Molecular Function terms: 10 
Plot generated for: R270X D18 vs WT D90  

===  06_DEG_GO  ===
Comparison: R270X D46 vs R270X D9

In [8]:
# List all Enrichr Excel files
files <- list.files(path = "06_DEG_GO", pattern = "\\.xlsx$", full.names = TRUE, recursive = TRUE)

# Mapping of sheet names to database names
sheet_mapping <- c(
  "GO_Biological_Process_2025" = "GO_Biological_Process_2025",
  "GO_Cellular_Component_2025" = "GO_Cellular_Component_2025",
  "GO_Molecular_Function_2025" = "GO_Molecular_Function_2025",
  "KEGG_2021_Human" = "KEGG_2021_Human",
  "Panther_2016" = "Panther_2016",
  "Reactome_2022" = "Reactome_2022",
  "RNAseq_DiseaseGene_DrugSigs_GEO" = "RNAseq_DiseaseGene_DrugSigs_GEO"
)

# Loop through each Excel file
for (file in files) {
  
  # Extract comparison name from filename
  comparison_name <- gsub("_results_enrichr_results", "", basename(file))
  comparison_name <- gsub("\\.xlsx$", "", comparison_name)
  comparison_name <- gsub("_", " ", comparison_name)

  cat("Comparison:", comparison_name, "\n")
  
  # Loop through each database sheet
  for (sheet_name in names(sheet_mapping)) {
    tryCatch({
      # Read the sheet
      db_terms <- read_excel(file, sheet = sheet_mapping[sheet_name])

      # Filter for significance 
      db_terms <- db_terms[db_terms$Adjusted.P.value < 0.05, ]
        
      # Skip if no data
      if (nrow(db_terms) == 0) {
        cat("No data for", sheet_name, "\n")
        next
      }

      pdf_dir <- file.path("00_FIGURES", sheet_name)
      if (!dir.exists(pdf_dir)) dir.create(pdf_dir, recursive = TRUE)

      # Define the PDF filename
      pdf_filename <- paste0("00_FIGURES/", sheet_name, "/", comparison_name, "_", sheet_name, ".pdf")
      
      # Create the plot and save it
      pdf(pdf_filename, height = 7, width = 17)
      p <- plotEnrich(db_terms, showTerms = 25, numChar = 75, y = "Count", orderBy = "Adjusted.P.value") +
        ggtitle(paste(sheet_name, ":", comparison_name))
      
      print(p)
      dev.off()
      
      # Print success message
      cat("Plot saved:", sheet_name, "->", pdf_filename, "\n")
      
    }, error = function(e) {
      # Handle any read errors
      cat("Error reading", sheet_name, ":", e$message, "\n")
    })
  }
}

Comparison: R270X D18 vs R270X D46 
No data for GO_Biological_Process_2025 
No data for GO_Cellular_Component_2025 
No data for GO_Molecular_Function_2025 
No data for KEGG_2021_Human 
No data for Panther_2016 
No data for Reactome_2022 
No data for RNAseq_DiseaseGene_DrugSigs_GEO 
Comparison: R270X D18 vs R270X D90 
No data for GO_Biological_Process_2025 
No data for GO_Cellular_Component_2025 
No data for GO_Molecular_Function_2025 
No data for KEGG_2021_Human 
No data for Panther_2016 
No data for Reactome_2022 
No data for RNAseq_DiseaseGene_DrugSigs_GEO 
Comparison: R270X D18 vs WT D18 
No data for GO_Biological_Process_2025 
No data for GO_Cellular_Component_2025 
No data for GO_Molecular_Function_2025 
No data for KEGG_2021_Human 
No data for Panther_2016 
No data for Reactome_2022 
No data for RNAseq_DiseaseGene_DrugSigs_GEO 
Comparison: R270X D18 vs WT D46 
No data for GO_Biological_Process_2025 
No data for GO_Cellular_Component_2025 
No data for GO_Molecular_Function_2025 
N

Warning message:
“`aes_string()` was deprecated in ggplot2 3.0.0.
ℹ Please use tidy evaluation idioms with `aes()`.
ℹ See also `vignette("ggplot2-in-packages")` for more information.
ℹ The deprecated feature was likely used in the enrichR package.
  Please report the issue to the authors.”


Plot saved: Panther_2016 -> 00_FIGURES/Panther_2016/R270X D46 vs WT D46_Panther_2016.pdf 
No data for Reactome_2022 
No data for RNAseq_DiseaseGene_DrugSigs_GEO 
Comparison: R270X D46 vs WT D90 
No data for GO_Biological_Process_2025 
No data for GO_Cellular_Component_2025 
No data for GO_Molecular_Function_2025 
No data for KEGG_2021_Human 
No data for Panther_2016 
No data for Reactome_2022 
No data for RNAseq_DiseaseGene_DrugSigs_GEO 
Comparison: R270X D90 vs WT D90 
No data for GO_Biological_Process_2025 
No data for GO_Cellular_Component_2025 
No data for GO_Molecular_Function_2025 
No data for KEGG_2021_Human 
No data for Panther_2016 
No data for Reactome_2022 
No data for RNAseq_DiseaseGene_DrugSigs_GEO 
Comparison: WT D18 vs R270X D46 
No data for GO_Biological_Process_2025 
No data for GO_Cellular_Component_2025 
No data for GO_Molecular_Function_2025 
No data for KEGG_2021_Human 
No data for Panther_2016 
No data for Reactome_2022 
No data for RNAseq_DiseaseGene_DrugSigs_GEO